# Machine learning pour la vulnérabilité côtière: comparaison critique de six modèles

**GGOSSS 2026 · Jour 2 · Instructeur: Nourdi Njutapvoui**

Question de décision: **le machine learning peut-il appuyer une carte de vulnérabilité côtière sans masquer l'incertitude ou un biais méthodologique?**

Ce TP utilise les données de vulnérabilité côtière du Cameroun provenant de `BD` et compare six algorithmes supervisés: SVM, Random Forest, ANN/MLP, Decision Tree, Logistic Regression et KNN.

## Objectifs pédagogiques

À la fin de ce TP, les participants seront capables de:

- configurer six algorithmes de classification dans scikit-learn;
- comparer les modèles avec l'accuracy, le F1 pondéré et le kappa de Cohen;
- détecter pourquoi des prédicteurs dérivés peuvent créer du leakage;
- comparer la validation aléatoire et la validation spatiale;
- expliquer pourquoi le meilleur score numérique n'est pas toujours le meilleur modèle scientifique.

In [ ]:
# Bloc de préparation GGOSSS 2026: importer les packages, définir le style visuel et localiser les données.
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, cohen_kappa_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

GGOSSS_COLORS = {
    "navy": "#17304f",
    "ocean": "#0077b6",
    "cyan": "#00a6a6",
    "sand": "#f2c14e",
    "coral": "#f05d5e",
    "green": "#2a9d8f",
}

plt.rcParams.update({
    "figure.figsize": (8, 4),
    "axes.grid": True,
    "axes.facecolor": "#fbfcfd",
    "axes.edgecolor": GGOSSS_COLORS["navy"],
    "axes.titleweight": "bold",
    "axes.titlesize": 12,
})

def find_data_dir():
    """Find the data folder in either the GitHub layout or the local instructor layout."""
    for candidate in [Path("../data"), Path("../Datasets"), Path("data"), Path("Datasets")]:
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Could not find data directory. Run from the notebook folder or the session folder.")

DATA_DIR = find_data_dir()
DATA_DIR

In [ ]:
# Bloc de chargement: lire la table nettoyée de vulnérabilité côtière dérivée de BD.
df = pd.read_csv(DATA_DIR / "cameroon_coastal_vulnerability_ml.csv")
print("Rows, columns:", df.shape)
df.head()

In [ ]:
# Bloc cartographique: convertir les coordonnées UTM en lon/lat et représenter le littoral camerounais.
# Cette carte sert à comprendre la logique spatiale avant la validation des algorithmes.
from matplotlib.patches import Rectangle
from pyproj import Transformer

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    CARTOPY_DISPONIBLE = True
except Exception:
    CARTOPY_DISPONIBLE = False

transformer = Transformer.from_crs("EPSG:32632", "EPSG:4326", always_xy=True)
lon, lat = transformer.transform(df["x_utm"].to_numpy(), df["y_utm"].to_numpy())
df_map = df.assign(longitude=lon, latitude=lat)

cameroon_places = pd.DataFrame(
    [
        {"Name": "Rio del Rey", "Lon": 8.78, "Lat": 4.55},
        {"Name": "Limbé", "Lon": 9.21, "Lat": 4.02},
        {"Name": "Douala", "Lon": 9.70, "Lat": 4.05},
        {"Name": "Kribi", "Lon": 9.91, "Lat": 2.94},
        {"Name": "Campo", "Lon": 9.82, "Lat": 2.38},
    ]
)
extent_detail = [
    df_map["longitude"].min() - 0.25,
    df_map["longitude"].max() + 0.25,
    df_map["latitude"].min() - 0.25,
    df_map["latitude"].max() + 0.25,
]
extent_regional = [7.2, 12.7, 1.0, 5.3]
sample_map = df_map.sample(n=min(2200, len(df_map)), random_state=42)

def ajouter_fond_geographique(ax, extent, titre):
    """Ajouter continent, océan, trait de côte, frontières et grille géographique."""
    if CARTOPY_DISPONIBLE:
        ax.set_extent(extent, crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.OCEAN, facecolor="#d8eef7", zorder=0)
        ax.add_feature(cfeature.LAND, facecolor="#f1efe6", edgecolor="#9ca3af", linewidth=0.25, zorder=1)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.9, edgecolor="#25313b", zorder=3)
        ax.add_feature(cfeature.BORDERS, linewidth=0.45, edgecolor="#6b7280", zorder=3)
        grille = ax.gridlines(draw_labels=True, linewidth=0.25, color="#6b7280", alpha=0.45)
        grille.top_labels = False
        grille.right_labels = False
    else:
        ax.set_xlim(extent[0], extent[1])
        ax.set_ylim(extent[2], extent[3])
        ax.set_facecolor("#d8eef7")
        ax.add_patch(Rectangle((9.0, 1.0), 3.7, 4.3, facecolor="#f1efe6", edgecolor="#9ca3af", linewidth=0.8))
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
    ax.set_title(titre, fontsize=11, weight="bold")

projection = ccrs.PlateCarree() if CARTOPY_DISPONIBLE else None
fig = plt.figure(figsize=(12, 5.8), constrained_layout=True)
ax_context = fig.add_subplot(1, 2, 1, projection=projection) if CARTOPY_DISPONIBLE else fig.add_subplot(1, 2, 1)
ax_detail = fig.add_subplot(1, 2, 2, projection=projection) if CARTOPY_DISPONIBLE else fig.add_subplot(1, 2, 2)
transform = ccrs.PlateCarree() if CARTOPY_DISPONIBLE else None
kwargs = {"transform": transform} if CARTOPY_DISPONIBLE else {}

# Carte régionale: situer le Cameroun côtier dans le Golfe de Guinée avec un vrai trait de côte.
ajouter_fond_geographique(ax_context, extent_regional, "Contexte régional - Golfe de Guinée")
ax_context.add_patch(Rectangle((extent_detail[0], extent_detail[2]), extent_detail[1] - extent_detail[0], extent_detail[3] - extent_detail[2],
                               fill=False, edgecolor=GGOSSS_COLORS["coral"], linewidth=2.0, transform=transform))
for nom, x, y in [
    ("Nigéria", 8.0, 4.45),
    ("Cameroun", 10.15, 4.55),
    ("Guinée équatoriale", 10.75, 1.35),
    ("Gabon", 11.55, 1.1),
    ("Océan Atlantique", 7.75, 1.45),
]:
    ax_context.text(x, y, nom, fontsize=9, color=GGOSSS_COLORS["navy"] if nom == "Cameroun" else "#4b5563", weight="bold" if nom == "Cameroun" else "normal", **kwargs)

# Carte détaillée: points ML, classes IVCI et séparation spatiale utilisée pour le test.
ajouter_fond_geographique(ax_detail, extent_detail, "Échantillons côtiers et séparation spatiale")
scatter = ax_detail.scatter(
    sample_map["longitude"],
    sample_map["latitude"],
    c=sample_map["vulnerability_class"],
    s=20,
    cmap="YlOrRd",
    vmin=1,
    vmax=5,
    alpha=0.78,
    edgecolor="none",
    zorder=5,
    **kwargs,
)
split_lon = df_map["longitude"].median()
ax_detail.plot([split_lon, split_lon], [extent_detail[2], extent_detail[3]], color=GGOSSS_COLORS["navy"], linestyle="--", linewidth=1.3, zorder=6, **kwargs)
ax_detail.text(split_lon + 0.015, extent_detail[3] - 0.12, "séparation spatiale", fontsize=8, color=GGOSSS_COLORS["navy"], **kwargs)
ax_detail.scatter(cameroon_places["Lon"], cameroon_places["Lat"], marker="^", color=GGOSSS_COLORS["navy"], s=52, zorder=7, **kwargs)
for _, row in cameroon_places.iterrows():
    ax_detail.text(row["Lon"] + 0.015, row["Lat"] + 0.015, row["Name"], fontsize=8, color="#111827", **kwargs)

cbar = fig.colorbar(scatter, ax=ax_detail, shrink=0.82)
cbar.set_label("Classe de vulnérabilité IVCI")
fig.suptitle("GGOSSS 2026 - Données ML côtières sur fond continental Natural Earth", fontsize=13, weight="bold")
plt.show()

In [ ]:
# Bloc de définition des variables: définir le jeu complet de prédicteurs et le jeu réduit pour discuter le leakage.
target = "vulnerability_class"
base_features = [
    "coastal_slope",
    "elevation_m",
    "shoreline_change_myr",
    "sea_level_anomaly_m",
    "tide_m",
    "significant_wave_height_m",
    "land_surface_temperature",
    "wind_speed_ms",
]
derived_index_features = ["physical_vulnerability_index", "socioeconomic_vulnerability_index"]
full_features = base_features + derived_index_features

X_full = df[full_features]
X_reduced = df[base_features]
y = df[target]

print("Class counts:")
print(y.value_counts().sort_index())
print("\nMissing values in full predictors:")
print(X_full.isna().sum().sort_values(ascending=False))

In [ ]:
# Bloc d'exploration: afficher la distribution des classes de vulnérabilité avant l'entraînement.
fig, ax = plt.subplots()
y.value_counts().sort_index().plot(kind="bar", ax=ax, color=GGOSSS_COLORS["ocean"])
ax.set_xlabel("Integrated vulnerability class")
ax.set_ylabel("Number of coastal points")
ax.set_title("GGOSSS 2026 · Cameroon coastal vulnerability classes")
plt.show()

In [ ]:
# Bloc de modélisation réutilisable: définir les six algorithmes avec imputation et standardisation lorsque nécessaire.
def build_models():
    return {
        "SVM": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), SVC(kernel="rbf", C=10, gamma="scale")),
        "RF": make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=150, random_state=42, class_weight="balanced")),
        "ANN_MLP": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), MLPClassifier(hidden_layer_sizes=(40, 20), max_iter=600, random_state=42, early_stopping=True)),
        "DT": make_pipeline(SimpleImputer(strategy="median"), DecisionTreeClassifier(max_depth=8, random_state=42, class_weight="balanced")),
        "LR": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced")),
        "KNN": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), KNeighborsClassifier(n_neighbors=7)),
    }

def evaluate_models(X_train, X_test, y_train, y_test):
    results = []
    predictions = {}
    fitted = {}
    for name, model in build_models().items():
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        predictions[name] = pred
        fitted[name] = model
        results.append({
            "model": name,
            "accuracy": accuracy_score(y_test, pred),
            "precision_macro": precision_score(y_test, pred, average="macro", zero_division=0),
            "recall_macro": recall_score(y_test, pred, average="macro", zero_division=0),
            "f1_weighted": f1_score(y_test, pred, average="weighted", zero_division=0),
            "kappa": cohen_kappa_score(y_test, pred),
        })
    metrics = pd.DataFrame(results).sort_values("f1_weighted", ascending=False)
    return metrics, predictions, fitted

list(build_models())

In [ ]:
# Bloc validation aléatoire: appliquer un split stratifié train/test avec tous les prédicteurs.
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y, test_size=0.30, random_state=42, stratify=y
)
metrics_full, predictions_full, fitted_full = evaluate_models(X_train, X_test, y_train, y_test)
metrics_full

In [ ]:
# Bloc comparaison des modèles: visualiser les six algorithmes avec la validation aléatoire.
fig, ax = plt.subplots(figsize=(9, 4))
metrics_full.set_index("model")[["accuracy", "f1_weighted", "kappa"]].plot(
    kind="bar", ax=ax, color=[GGOSSS_COLORS["ocean"], GGOSSS_COLORS["green"], GGOSSS_COLORS["sand"]]
)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title("GGOSSS 2026 · Six-model comparison with all predictors")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Bloc test de leakage: retirer les indices dérivés de vulnérabilité et comparer à nouveau les scores.
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reduced, y, test_size=0.30, random_state=42, stratify=y
)
metrics_reduced, predictions_reduced, fitted_reduced = evaluate_models(X_train_r, X_test_r, y_train_r, y_test_r)

comparison = metrics_full[["model", "f1_weighted", "kappa"]].merge(
    metrics_reduced[["model", "f1_weighted", "kappa"]],
    on="model",
    suffixes=("_all_predictors", "_no_derived_indices"),
)
comparison["f1_drop"] = comparison["f1_weighted_all_predictors"] - comparison["f1_weighted_no_derived_indices"]
comparison.sort_values("f1_drop", ascending=False)

In [ ]:
# Bloc visualisation du leakage: montrer à quel point la performance dépend des indicateurs dérivés proches de l'AHP.
plot = comparison.set_index("model")[["f1_weighted_all_predictors", "f1_weighted_no_derived_indices"]]
ax = plot.plot(kind="bar", figsize=(9, 4), color=[GGOSSS_COLORS["coral"], GGOSSS_COLORS["cyan"]])
ax.set_ylim(0, 1.05)
ax.set_ylabel("Weighted F1-score")
ax.set_title("GGOSSS 2026 · Leakage sensitivity test")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Bloc validation spatiale: entraîner sur une moitié du littoral et tester sur l'autre.
# Cette validation est plus stricte qu'un split aléatoire car les points côtiers proches sont spatialement dépendants.
longitude_cut = df["x_utm"].median()
train_mask = df["x_utm"] <= longitude_cut
test_mask = ~train_mask

X_train_s = X_reduced.loc[train_mask]
X_test_s = X_reduced.loc[test_mask]
y_train_s = y.loc[train_mask]
y_test_s = y.loc[test_mask]

metrics_spatial, predictions_spatial, fitted_spatial = evaluate_models(X_train_s, X_test_s, y_train_s, y_test_s)
print("Training points:", len(X_train_s), "Testing points:", len(X_test_s))
metrics_spatial

In [ ]:
# Bloc comparaison des validations: comparer validation aléatoire et validation spatiale pour le jeu réduit.
validation_comparison = metrics_reduced[["model", "f1_weighted", "kappa"]].merge(
    metrics_spatial[["model", "f1_weighted", "kappa"]],
    on="model",
    suffixes=("_random", "_spatial"),
)
validation_comparison["f1_random_minus_spatial"] = validation_comparison["f1_weighted_random"] - validation_comparison["f1_weighted_spatial"]
validation_comparison.sort_values("f1_random_minus_spatial", ascending=False)

In [ ]:
# Bloc diagnostic des erreurs: examiner la matrice de confusion du meilleur modèle en validation spatiale.
best_spatial_model = metrics_spatial.iloc[0]["model"]
print("Best model under spatial validation:", best_spatial_model)
cm = confusion_matrix(y_test_s, predictions_spatial[best_spatial_model], labels=sorted(y.unique()))
disp = ConfusionMatrixDisplay(cm, display_labels=sorted(y.unique()))
disp.plot(cmap="Blues", values_format="d")
plt.title(f"GGOSSS 2026 · Spatial confusion matrix: {best_spatial_model}")
plt.show()

In [ ]:
# Bloc planche comparative: reproduire une figure de type IVCI observe + predictions ML.
# Chaque algorithme est représenté comme un profil côtier décalé afin de comparer visuellement les classes.
from matplotlib.lines import Line2D
from sklearn.metrics import accuracy_score

couleurs_classes = {
    1: "#1a7f00",
    2: "#65a800",
    3: "#fff200",
    4: "#ff9f1c",
    5: "#e31a1c",
}
etiquettes_classes = {
    1: "Très faible",
    2: "Faible",
    3: "Modérée",
    4: "Forte",
    5: "Très forte",
}

# Les modèles spatiaux ont été entraînés sur une moitié du littoral; on prédit ensuite tout le linéaire pour visualiser les différences.
predictions_carte = df_map[["longitude", "latitude", target]].copy()
for nom_modele, modele in fitted_spatial.items():
    predictions_carte[f"prediction_{nom_modele}"] = modele.predict(X_reduced)

donnees_planche = predictions_carte.sort_values(["latitude", "longitude"]).iloc[::2].copy()
modeles_planche = [
    ("prediction_RF", "RF"),
    ("prediction_SVM", "SVM"),
    ("prediction_LR", "LR"),
    ("prediction_DT", "DT"),
    ("prediction_KNN", "KNN"),
    ("prediction_ANN_MLP", "ANN-MLPC"),
]

fig, (ax_observe, ax_modeles) = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={"width_ratios": [1.15, 2.4]})

# Carte observée: conserver les vraies coordonnées longitude/latitude et les localités principales.
ax_observe.scatter(
    donnees_planche["longitude"],
    donnees_planche["latitude"],
    c=donnees_planche[target].astype(int).map(couleurs_classes),
    s=28,
    edgecolors="none",
    linewidths=0,
)
localites = pd.DataFrame([
    {"nom": "Rio del Rey", "lon": 8.78, "lat": 4.55},
    {"nom": "Limbe", "lon": 9.21, "lat": 4.02},
    {"nom": "Douala", "lon": 9.70, "lat": 4.05},
    {"nom": "Kribi", "lon": 9.91, "lat": 2.94},
    {"nom": "Campo", "lon": 9.82, "lat": 2.38},
])
for _, ligne in localites.iterrows():
    ax_observe.scatter(ligne["lon"], ligne["lat"], marker="^", color=GGOSSS_COLORS["navy"], s=46)
    ax_observe.text(ligne["lon"] + 0.02, ligne["lat"] + 0.015, ligne["nom"], fontsize=8, weight="bold")
ax_observe.set_xlim(7.75, 10.12)
ax_observe.set_ylim(2.15, 4.85)
ax_observe.set_xlabel("Longitude")
ax_observe.set_ylabel("Latitude")
ax_observe.set_title("IVCI observé")
ax_observe.set_facecolor("#d8eef7")

# Profils prédits: normaliser la longitude pour juxtaposer les six algorithmes sans superposition.
span_lon = donnees_planche["longitude"].max() - donnees_planche["longitude"].min()
centre_lat = (donnees_planche["latitude"].min() + donnees_planche["latitude"].max()) / 2
for i, (colonne, etiquette) in enumerate(modeles_planche):
    x_norm = (donnees_planche["longitude"] - donnees_planche["longitude"].min()) / span_lon
    x_panel = i * 1.25 + x_norm * 0.58
    y_panel = (donnees_planche["latitude"] - centre_lat) * 0.96 + 3.45
    ax_modeles.scatter(
        x_panel,
        y_panel,
        c=donnees_planche[colonne].astype(int).map(couleurs_classes),
        s=18,
        edgecolors="#333333",
        linewidths=0.15,
    )
    acc = accuracy_score(donnees_planche[target], donnees_planche[colonne])
    f1 = f1_score(donnees_planche[target], donnees_planche[colonne], average="weighted", zero_division=0)
    ax_modeles.text(i * 1.25 + 0.29, 2.25, etiquette, fontsize=16, ha="center", fontfamily="serif")
    ax_modeles.text(i * 1.25 + 0.29, 2.10, f"Acc={acc:.2f} | F1={f1:.2f}", fontsize=8, ha="center", color="#444444")

ax_modeles.set_ylim(2.05, 4.9)
ax_modeles.set_xlim(-0.18, 1.25 * 5 + 0.78)
ax_modeles.axis("off")

legende = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=couleur, markeredgecolor="#333333", markersize=8, label=f"{classe} - {etiquettes_classes[classe]}")
    for classe, couleur in couleurs_classes.items()
]
ax_observe.legend(handles=legende, title="Classe IVCI", loc="lower left", fontsize=8)
fig.suptitle("GGOSSS 2026 - IVCI observé et classes prédites par algorithme", fontsize=14, weight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Bloc interprétation: utiliser l'importance Random Forest du modèle réduit pour discuter les facteurs physiques.
rf = fitted_reduced["RF"].named_steps["randomforestclassifier"]
importance = pd.Series(rf.feature_importances_, index=base_features).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(8, 5))
importance.plot(kind="barh", ax=ax, color=GGOSSS_COLORS["cyan"])
ax.set_title("GGOSSS 2026 · Random Forest feature importance without derived indices")
ax.set_xlabel("Relative importance")
plt.show()

## Mini-challenge: choix d'un modèle pour une décision côtière

Chaque groupe doit produire une réponse concise:

1. Choose one model for a coastal vulnerability screening map.
2. Justify the choice using metrics and interpretability.
3. State whether you trust the random validation or the spatial validation more.
4. Identify one leakage risk.
5. Name one field observation that would improve the model.

Une bonne réponse ne choisit pas simplement le modèle avec le meilleur score; elle explique le risque scientifique.